# GP Emulator Calibration

Calibrate Heston parameters `(v0, kappa, theta, sigma_v, rho)` to own calculated implied volatilities with the saved GP emulator. The optimizer searches for the parameters that minimize total squared IV error across the option chain.

In [1]:
from __future__ import annotations

import csv
import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

PARAMETER_COLUMNS = ["v0", "kappa", "theta", "sigma_v", "rho"]
FEATURE_COLUMNS = [*PARAMETER_COLUMNS, "K", "T"]

## Settings

In [2]:
MODEL_PATH = Path("outputs/gp_emulator/gp_emulator_model.npz")
MARKET_DATA_PATH = Path("data/spy_options_clean_with_iv.csv")
TRAINING_DATA_PATH = Path("data/simulated_training_data.csv")
OUTPUT_DIR = Path("outputs/gp_calibration")

MAXITER = 300
FTOL = 1e-12
PREDICTION_CHUNK_SIZE = 4096

# Match the GP training scale: S0 was 100 in the synthetic data.
NORMALIZE_STRIKE = True

# Use "all", "call", or "put".
OPTION_TYPE = "all"

# Keep only contracts inside the GP training grid if this is True.
DOMAIN_ONLY = False
MIN_K, MAX_K = 85.0, 115.0
MIN_T, MAX_T = 0.019178082191780823, 0.2465753424657534

## GP Emulator Loader

In [3]:
@dataclass(frozen=True)
class GPEmulator:
    x_train_scaled: np.ndarray
    x_mean: np.ndarray
    x_scale: np.ndarray
    y_mean: float
    y_scale: float
    length_scales: np.ndarray
    signal_std: float
    alpha: np.ndarray

    @classmethod
    def load(cls, path: Path) -> "GPEmulator":
        # Load the saved GP state and scalers from training.
        model = np.load(path)
        feature_columns = [str(name) for name in model["feature_columns"]]
        if feature_columns != FEATURE_COLUMNS:
            raise ValueError(f"{path} has feature columns {feature_columns}; expected {FEATURE_COLUMNS}")

        return cls(
            x_train_scaled=np.asarray(model["x_train_scaled"], dtype=float),
            x_mean=np.asarray(model["x_mean"], dtype=float),
            x_scale=np.asarray(model["x_scale"], dtype=float),
            y_mean=float(np.asarray(model["y_mean"]).ravel()[0]),
            y_scale=float(np.asarray(model["y_scale"]).ravel()[0]),
            length_scales=np.asarray(model["length_scales"], dtype=float),
            signal_std=float(np.asarray(model["signal_std"])),
            alpha=np.asarray(model["alpha"], dtype=float),
        )

    def predict_iv(self, features: np.ndarray, chunk_size: int = 4096) -> np.ndarray:
        # Standardize inputs exactly as they were standardized for GP training.
        x_query_scaled = (features - self.x_mean) / self.x_scale
        predictions = np.empty(x_query_scaled.shape[0], dtype=float)

        # Pre-scale the training points once, then score query points in chunks.
        x_train_scaled = self.x_train_scaled / self.length_scales
        x_train_norm = np.sum(x_train_scaled * x_train_scaled, axis=1)[None, :]

        for start in range(0, x_query_scaled.shape[0], chunk_size):
            stop = min(start + chunk_size, x_query_scaled.shape[0])
            x_chunk = x_query_scaled[start:stop] / self.length_scales
            x_chunk_norm = np.sum(x_chunk * x_chunk, axis=1)[:, None]
            distances = x_chunk_norm + x_train_norm - 2.0 * x_chunk @ x_train_scaled.T
            kernel = self.signal_std**2 * np.exp(-0.5 * np.maximum(distances, 0.0))
            predictions[start:stop] = kernel @ self.alpha

        return predictions * self.y_scale + self.y_mean

## Market Data and Objective

In [4]:
def load_market_data(path: Path, normalize_strike: bool) -> pd.DataFrame:
    # Read own calculated IVs and convert expiry dates into year fractions.
    required = {"strike", "expiry", "pull_date", "iv_own", "S0"}
    market = pd.read_csv(path)
    missing = sorted(required - set(market.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    expiry = pd.to_datetime(market["expiry"], errors="coerce")
    pull_date = pd.to_datetime(market["pull_date"], errors="coerce")
    market = market.assign(
        T=(expiry - pull_date).dt.days / 365.0,
        market_iv=pd.to_numeric(market["iv_own"], errors="coerce"),
        K=pd.to_numeric(market["strike"], errors="coerce"),
        S0=pd.to_numeric(market["S0"], errors="coerce"),
    )

    if normalize_strike:
        # Convert real strikes onto the S0=100 strike scale used by the GP.
        market["K"] = 100.0 * market["K"] / market["S0"]

    finite = np.isfinite(market[["K", "T", "market_iv"]]).all(axis=1)
    positive = (market["T"] > 0.0) & (market["market_iv"] > 0.0)
    return market.loc[finite & positive].copy()


def parameter_bounds(training_data: Path | None) -> list[tuple[float, float]]:
    # Use the GP training range as the optimizer bounds.
    if training_data is None:
        return [
            (0.01, 0.25),
            (0.10, 5.00),
            (0.01, 0.25),
            (0.05, 1.00),
            (-0.90, 0.00),
        ]

    training = pd.read_csv(training_data, usecols=PARAMETER_COLUMNS)
    return [
        (float(training[column].min()), float(training[column].max()))
        for column in PARAMETER_COLUMNS
    ]


def initial_guess(bounds: list[tuple[float, float]]) -> np.ndarray:
    return np.array([(low + high) / 2.0 for low, high in bounds], dtype=float)


def make_feature_matrix(params: np.ndarray, market_k: np.ndarray, market_t: np.ndarray) -> np.ndarray:
    # Repeat the candidate parameters across all market contracts.
    features = np.empty((market_k.size, len(FEATURE_COLUMNS)), dtype=float)
    features[:, : len(PARAMETER_COLUMNS)] = params
    features[:, len(PARAMETER_COLUMNS)] = market_k
    features[:, len(PARAMETER_COLUMNS) + 1] = market_t
    return features

## Run Calibration

In [5]:
# Load the GP and market contracts.
gp = GPEmulator.load(MODEL_PATH)
market = load_market_data(MARKET_DATA_PATH, normalize_strike=NORMALIZE_STRIKE)

# Optional filters for option type and GP domain.
if OPTION_TYPE != "all" and "type" in market.columns:
    market = market.loc[market["type"].str.lower() == OPTION_TYPE].copy()

if DOMAIN_ONLY:
    in_domain = market["K"].between(MIN_K, MAX_K) & market["T"].between(MIN_T, MAX_T)
    market = market.loc[in_domain].copy()

if market.empty:
    raise ValueError("No market contracts remain after filtering.")

market_k = market["K"].to_numpy(dtype=float)
market_t = market["T"].to_numpy(dtype=float)
market_iv = market["market_iv"].to_numpy(dtype=float)
bounds = parameter_bounds(TRAINING_DATA_PATH)


def objective(params: np.ndarray) -> float:
    # Predict IVs for this parameter guess and sum squared IV errors.
    features = make_feature_matrix(params, market_k, market_t)
    predicted_iv = gp.predict_iv(features, chunk_size=PREDICTION_CHUNK_SIZE)
    residual = predicted_iv - market_iv
    return float(np.sum(residual * residual))


# Run the same optimizer style used in the Kou calibration paper.
started_at = time.perf_counter()
result = minimize(
    objective,
    x0=initial_guess(bounds),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": MAXITER, "ftol": FTOL},
)
wall_clock_seconds = time.perf_counter() - started_at

# Recompute predictions at the fitted parameters for reporting.
fitted_params = np.asarray(result.x, dtype=float)
fitted_features = make_feature_matrix(fitted_params, market_k, market_t)
predicted_iv = gp.predict_iv(fitted_features, chunk_size=PREDICTION_CHUNK_SIZE)
residual = predicted_iv - market_iv
squared_error = residual * residual
total_error = float(np.sum(squared_error))

summary = {
    "success": bool(result.success),
    "message": str(result.message),
    "contracts": int(len(market)),
    "evaluations": int(result.nfev),
    "fitted_parameters": {
        column: float(value)
        for column, value in zip(PARAMETER_COLUMNS, fitted_params)
    },
    "total_calibration_error": total_error,
    "rmse": float(np.sqrt(np.mean(squared_error))),
    "mae": float(np.mean(np.abs(residual))),
    "wall_clock_seconds": float(wall_clock_seconds),
    "bounds": {column: list(bound) for column, bound in zip(PARAMETER_COLUMNS, bounds)},
    "market_data": str(MARKET_DATA_PATH),
    "model": str(MODEL_PATH),
    "normalized_strike": bool(NORMALIZE_STRIKE),
    "domain_only": bool(DOMAIN_ONLY),
    "option_type": OPTION_TYPE,
}

summary

{'success': True,
 'message': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH',
 'contracts': 899,
 'evaluations': 516,
 'fitted_parameters': {'v0': 0.02742675607740627,
  'kappa': 1.2780702425272705,
  'theta': 0.2375397622957753,
  'sigma_v': 0.9993863554271026,
  'rho': -0.9495768662885722},
 'total_calibration_error': 9.438988881290841,
 'rmse': 0.10246673342412406,
 'mae': 0.05480427936566043,
 'wall_clock_seconds': 12.270436708990019,
 'bounds': {'v0': [0.0100207958590628, 0.2498920421524241],
  'kappa': [0.1001058627817415, 4.998386022881946],
  'theta': [0.0101229951136501, 0.2499243545643585],
  'sigma_v': [0.0501999396475361, 0.9993863554271026],
  'rho': [-0.9495768662885722, -0.0005757410491344]},
 'market_data': 'data/spy_options_clean_with_iv.csv',
 'model': 'outputs/gp_emulator/gp_emulator_model.npz',
 'normalized_strike': True,
 'domain_only': False,
 'option_type': 'all'}

## Save Results

In [6]:
# Save one row per contract with fitted IV and residuals.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = market.copy()
for column, value in zip(PARAMETER_COLUMNS, fitted_params):
    predictions[f"fitted_{column}"] = value

predictions["gp_predicted_iv"] = predicted_iv
predictions["iv_residual"] = residual
predictions["iv_squared_error"] = squared_error
predictions.to_csv(OUTPUT_DIR / "calibration_predictions.csv", index=False)

# Save the fitted parameters and error metrics in JSON and CSV formats.
with (OUTPUT_DIR / "calibration_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
    f.write("\n")

with (OUTPUT_DIR / "calibration_summary.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        *PARAMETER_COLUMNS,
        "total_calibration_error",
        "rmse",
        "mae",
        "wall_clock_seconds",
        "contracts",
        "evaluations",
        "success",
        "message",
    ])
    writer.writerow([
        *[summary["fitted_parameters"][column] for column in PARAMETER_COLUMNS],
        summary["total_calibration_error"],
        summary["rmse"],
        summary["mae"],
        summary["wall_clock_seconds"],
        summary["contracts"],
        summary["evaluations"],
        summary["success"],
        summary["message"],
    ])

print(f"Saved calibration summary to {OUTPUT_DIR / 'calibration_summary.csv'}")
print(f"Saved per-contract predictions to {OUTPUT_DIR / 'calibration_predictions.csv'}")

Saved calibration summary to outputs/gp_calibration/calibration_summary.csv
Saved per-contract predictions to outputs/gp_calibration/calibration_predictions.csv
